# Cuaderno asociado al TFG

Este cuaderno forma parte del repositorio asociado al Trabajo de Fin de Grado **“Impacto de las Telecomunicaciones en la Agricultura 4.0”**.

Por motivos de confidencialidad, los archivos de datos reales no se incluyen en el repositorio. El código se mantiene como referencia metodológica y está preparado para trabajar con archivos Excel equivalentes ubicados en la carpeta `Datos/`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#codigo base para maiz
# 1) Cargar
df = pd.read_excel("../Datos/Datos_cultivos.xlsx")

# 2) Filtrar por cultivo (SIN mezclar)
maiz = df[df["Cultivo"].str.lower().str.contains("maiz")].copy()

# 3) Índice tecnológico (nivel de sistema)
tech_cols = ["GPS","RTK","ISOBUS","Siembra_variable","Abono_variable","Corte_tramos"]
maiz["tech_index"] = maiz[tech_cols].sum(axis=1)

# 4) KPIs derivados 
maiz["Produccion_total_t"] = maiz["Rend_t_ha"] * maiz["Superficie"]
maiz["Horas_totales_h"] = maiz["Superficie"] / maiz["Productividad_ha_h"]

maiz["Gasoil_total_L"] = maiz["Consumo_L_ha"] * maiz["Superficie"]
maiz["Gasoil_L_por_t"] = maiz["Consumo_L_ha"] / maiz["Rend_t_ha"]  # KPI estrella

maiz["Rango_semillas"] = maiz["Semillas_max_ha"] - maiz["Semillas_min_ha"]
maiz["Rango_abono"] = maiz["Abono_max_ha"] - maiz["Abono_min_ha"]

maiz["Var_rel_semillas"] = maiz["Rango_semillas"] / maiz["Semillas_ha"]
maiz["Var_rel_abono"] = maiz["Rango_abono"] / maiz["Abono_ha"]

# 5) Tabla resumen rápida
kpis = [
    "Rend_t_ha","Productividad_ha_h","Horas_totales_h",
    "Consumo_L_ha","Gasoil_L_por_t",
    "Semillas_ha","Abono_ha","Fitosanitarios_ha",
    "Var_rel_semillas","Var_rel_abono",
    "Pasadas_ha"
]

print("Filas maíz:", len(maiz))
display(maiz[["Año","Parcela","Superficie","tech_index"] + kpis].sort_values(["Parcela","Año"]))

In [ ]:
#comparativas por tecnologia 
def compare_flag(df_crop, flag, kpis):
    out = df_crop.groupby(flag)[kpis].agg(["mean","median","count"]).round(3)
    return out

display(compare_flag(maiz, "RTK", kpis))
display(compare_flag(maiz, "Siembra_variable", kpis))
display(compare_flag(maiz, "Abono_variable", kpis))
display(compare_flag(maiz, "ISOBUS", kpis))
display(compare_flag(maiz, "Corte_tramos", kpis))


In [ ]:
#correlaciones
corr_cols = tech_cols + ["tech_index"] + kpis
corr = maiz[corr_cols].corr(numeric_only=True)

# Para ver qué mueve más la productividad y el consumo L/t:
targets = ["Productividad_ha_h","Gasoil_L_por_t","Semillas_ha","Abono_ha","Fitosanitarios_ha","Rend_t_ha"]
display(corr[targets].sort_values(by="Gasoil_L_por_t", ascending=True).round(3))


In [ ]:
#Convertir resultados en impactos relativos
def impact_vs_baseline(df, flag, kpis):
    base = df[df[flag] == 0][kpis].mean()
    tech = df[df[flag] == 1][kpis].mean()
    impact = (tech - base) / base * 100
    return impact.round(1)

impact_RTK = impact_vs_baseline(maiz, "RTK", [
    "Productividad_ha_h",
    "Consumo_L_ha",
    "Gasoil_L_por_t",
    "Semillas_ha",
    "Abono_ha",
    "Fitosanitarios_ha"
])

impact_RTK


#esto nos pude dar algo como: “El uso de RTK reduce el consumo energético específico en un X%”.

In [ ]:
#estabilidad del sistema
def variability_analysis(df, flag, metric):
    return df.groupby(flag)[metric].agg(["mean","std","count"]).round(3)

variability_analysis(maiz, "RTK", "Gasoil_L_por_t")
variability_analysis(maiz, "RTK", "Productividad_ha_h")


#Si la std baja con tecnología → sistema más robusto.

In [ ]:
#escenarios tecnológicos

#crear niveles
maiz["tech_level"] = pd.cut(
    maiz["tech_index"],
    bins=[0,2,4,6],
    labels=["Bajo","Medio","Alto"]
)

#Comparar
maiz.groupby("tech_level")[[
    "Productividad_ha_h",
    "Gasoil_L_por_t",
    "Semillas_ha",
    "Abono_ha",
    "Fitosanitarios_ha"
]].agg(["mean","median","count"]).round(2)



bajo: poca tecnologia
medio: algunas, no todas. Algunos años que por algún motivo no hemos podido utilizar algunas de las tecnologias
alto: todas las tecnologias usadas

Conclusion: a mayor nivel tecnológico, mayor rendimiento operativo del sistema
El sistema con mayor nivel tecnológico, requiere significativamente menos energía por unidad de producción.
el numero de semillas usadas: 
abono medio utilizado
reduccion de fitosanitarios



In [ ]:
#regresión OLS.
#¿que tecnologías explican mejor la reducción del consumo energético específico?


import statsmodels.api as sm

X = maiz[["RTK","ISOBUS","Corte_tramos","Siembra_variable","Abono_variable"]]
X = sm.add_constant(X)

y = maiz["Gasoil_L_por_t"]

model = sm.OLS(y, X).fit()
print(model.summary())


Conclusiones del sistema OLS: 

variable dependiente Gasoil_L_por_t

p=0,002 es significativo
EL uso de RTK contribuye de forma significativa a reducir el consumo energético por tonelada

Isobus: p=0,002 también tiene un impacto directo en la eficiencia energética

Corte tramos: p=0,0001 muy significativo
El corte automático es la tecnología individual con mayor impacto en la reducción del consumo específico 

tanto siembra como abono variable: no tienen tan buena p=0,25. 
estas tecnologías no reducen directamente el consumo energético, sino que actúan sobre la distribución de insumos. esto es lo esperable. 



In [ ]:
#A igualdad de superficie, qué tecnologías están asociadas a una mayor producción total?

#variable dependiente
maiz["Produccion_total_t"] = maiz["Rend_t_ha"] * maiz["Superficie"]


X = maiz[[
    "Superficie",
    "RTK",
    "ISOBUS",
    "Corte_tramos",
    "Siembra_variable",
    "Abono_variable"
]]

X = sm.add_constant(X)
y = maiz["Produccion_total_t"]

model_prod = sm.OLS(y, X).fit()
print(model_prod.summary())



A igualdad de superficie, qué tecnologías están asociadas a una mayor producción total

superficie: como era de esperar p<0,001, cada hectarea aporta 16 toneladas de producción mas

De las tecnologías: 

corte por tramos: 0=0,042 significativo
el uso de corte de tramos está asociado a un incremento significativo de la producción total

RTK e Isobus
p=0,61, no son significativos en este modelo, algo lógico. Estas dos no aumentan directamente la producción total, sino que actúan como tecnologías habilitadoras de eficiencia y control. 

Siembra y abono variable
p>0,25, estas tecnologías no buscan maximizar toneladas, sino optimizar la distribución de insumos. 

El warning de multicolinealidad dice que: RTK, ISOBUS y otras tecnologías suelen aparecer juntas

In [ ]:
#que tecnologías están asociadas a un mayor rendimiento del terreno, independientemente del tamaño de la parcela

# Variables explicativas (solo tecnología)
X = maiz[
    [
        "RTK",
        "ISOBUS",
        "Corte_tramos",
        "Siembra_variable",
        "Abono_variable"
    ]
]

X = sm.add_constant(X)

# Variable dependiente
y = maiz["Rend_t_ha"]

# Modelo OLS explicativo
model_rend = sm.OLS(y, X).fit()

print(model_rend.summary())

conclusiones 

El rendimiento por hectárea mejora principalmente gracias a tecnologías que reducen solapes y pérdidas (corte por tramos),mientras que el resto de tecnologías actúan sobre la eficiencia y el control del sistema, no directamente sobre el rendimiento. 

El rendimiento base del sistema, sin tecnologías avanzadas, se situa en 13,4 t/ha


Corte por tramos: 
coef= 1,78 t/ha
p=0,027 significativo
A igualdad de condiciones, el uso de corte automático por tramos está asociado a un incremento medio del rendimiento por hectárea aproximadamente 1,8 t/ha


RTK e ISOBUS
p=0,69, no muy significativo. No incrementan directamente el rendimiento por hectárea, sino que permiten la correcta ejecución de otras tecnologías y mejora la eficiencia operativa. 

De igual manera la siembra y abono variable, no buscan aumentar el rendimiento máximo, sino optimizar la distribución de insumos y reducir desperdicios



In [ ]:
#variable dependiente: gasoil real L por tonelada: (consumo por pasada * nº pasadas)/rendimiento
#Energía total consumida por tonelada producida


maiz["Consumo_real_L_ha"] = maiz["Consumo_L_ha"] * maiz["Pasadas_ha"]
maiz["Gasoil_real_L_por_t"] = maiz["Consumo_real_L_ha"] / maiz["Rend_t_ha"]

X = maiz[
    ["RTK", "ISOBUS", "Corte_tramos", "Siembra_variable", "Abono_variable"]
]
X = sm.add_constant(X)

y = maiz["Gasoil_real_L_por_t"]

model_energy_real = sm.OLS(y, X).fit()
print(model_energy_real.summary())

Considerando el consumo total acumulado por campaña, las tecnologísa de Agricultura 4.0 permiten una reducción significativa del consumo energético específico por tonelada producida

R^2= 0,944, R^2 ajustado=0,928 p-valor del modelo <<0,001
El modelo explica la mayor parte de la variabilidad del consumo energético específico, lo que indica que las tecnologísa analizadas están fuertemente relacionadas con la eficiencia global del sistema

Corte por tramos: 
coef=-0,9987, p=0,004 muy significativo
El uso de corte automático por tramos reduce de forma significativa el consumo energético total por tonelada producida, con un impacto cercano a 1 L/t.

RTK:
coef = −0.460  
p = 0.033 oEl uso de RTK contribuye de forma significativa a la reducción del consumo energético específico, gracias a una mayor precisión en el guiado y a la reducción de recorridos redundantes. RTK no trabaja solo, pero sin él el sistema no funciona igual.

ISOBUS: 
coef = −0.460  
p = 0.033  significativoLa comunicación estandarizada entre tractor y apero permite una ejecución más eficiente de las labores, reduciendo el consumo energético total por tonelada producida.

RTK e ISOBUS aparecen con el mismo peso, lo cual es totalmente lógico:
uno posiciona, el otro ejecuta

Siembra Variable y abono variable: 

p aprox 0.25   no significativaLa siembra variable no tiene un impacto directo significativo sobre el consumo energético total, ya que su función principal es la optimización de la distribución de semillas, no la reducción del número de pasadas ni del consumo mecánico.

La dosificación variable de fertilizante influye principalmente en el uso de insumos y en la sostenibilidad, pero no en el consumo energético asociado a las operaciones de campo.


const = 7,48 L/t
En ausencia de tecnologías avanzadas, el sistema presenta un consumo energético específico elevado, cercano a 7.5 L de gasoil por tonelada producida. Luego, cada tecnología va reduciendo ese valor. 

Considerando el consumo total acumulado por campaña, las tecnologías de Agricultura 4.0 permiten una reducción significativa del consumo energético específico por tonelada producida.






TOMATE

In [ ]:
#data set tomates

tomate = df[df["Cultivo"].str.lower().str.contains("tomate")].copy()
print("Filas tomate:", len(tomate))


In [ ]:
tomate["Produccion_total_t"] = tomate["Rend_t_ha"] * tomate["Superficie"]
tomate["Horas_totales_h"] = tomate["Superficie"] / tomate["Productividad_ha_h"]

tomate["Gasoil_L_por_t"] = tomate["Consumo_L_ha"] / tomate["Rend_t_ha"]

tomate["tech_index"] = tomate[
    ["GPS","RTK","ISOBUS","Siembra_variable","Abono_variable","Corte_tramos"]
].sum(axis=1)


In [ ]:
tomate[[
    "Rend_t_ha",
    "Productividad_ha_h",
    "Consumo_L_ha",
    "Gasoil_L_por_t",
    "Semillas_ha",
    "Abono_ha",
    "Fitosanitarios_ha"
]].describe().round(2)


In [ ]:
#Escenarios tecnológicos
#definir niveles
tomate["tech_level"] = pd.cut(
    tomate["tech_index"],
    bins=[0,2,4,6],
    labels=["Bajo","Medio","Alto"]
)


In [ ]:
#comparar kpi
tomate.groupby("tech_level")[[
    "Productividad_ha_h",
    "Gasoil_L_por_t",
    "Semillas_ha",
    "Abono_ha",
    "Fitosanitarios_ha"
]].agg(["mean","median","count"]).round(2)


In [ ]:
#variable dependiente: Gasoil_L_por_t
import statsmodels.api as sm

X = tomate[
    ["RTK","ISOBUS","Corte_tramos","Siembra_variable","Abono_variable"]
]
X = sm.add_constant(X)

y = tomate["Gasoil_L_por_t"]

model_energy_tom = sm.OLS(y, X).fit()
print(model_energy_tom.summary())


In [ ]:
#Produccion total controlando superficie

X = tomate[
    ["Superficie","RTK","ISOBUS","Corte_tramos","Siembra_variable","Abono_variable"]
]
X = sm.add_constant(X)

y = tomate["Produccion_total_t"]

model_prod_tom = sm.OLS(y, X).fit()
print(model_prod_tom.summary())


In [ ]:
#rendimiento por hectárea
X = tomate[
    ["RTK","ISOBUS","Corte_tramos","Siembra_variable","Abono_variable"]
]
X = sm.add_constant(X)

y = tomate["Rend_t_ha"]

model_rend_tom = sm.OLS(y, X).fit()
print(model_rend_tom.summary())


En el cultivo de tomate, un mayor nivel tecnológico se asocia a una mejora clara de la eficiencia operativa y energética, así como a una reducción del uso de insumos, sin modificar la dosis de semillas ya que la siembra variable no se aplica en los tomates

En tomate, las tecnologías de posicionamiento, comunicación y control están claramente asociadas a una reducción del consumo energético específico por tonelada producida.

Ninguna variable tecnológica es significativa

En tomate, no se observa un impacto directo y estadísticamente significativo de las tecnologías analizadas sobre la producción total, una vez considerada la escala de explotacion.
Esto era algo esperable en un cultivo muy condicionado mayormente por el clima, según la campaña y el manejo del agricultor de la explotación.

No todo el impacto consiste en producir más toneladas, sino en el ahorro de insumos

En el cultivo de tomate, la implantación de tecnologías de Agricultura 4.0 se traduce principalmente en mejoras de eficiencia operativa y energética, con reducciones significativas del consumo de gasoil por tonelada producida.


GRAFICAS CLAVE

In [ ]:
#Preparar las variables definitivas
# === MAÍZ ===
maiz["Consumo_real_L_ha"] = maiz["Consumo_L_ha"] * maiz["Pasadas_ha"]
maiz["Gasoil_real_L_por_t"] = maiz["Consumo_real_L_ha"] / maiz["Rend_t_ha"]

# === TOMATE ===
tomate["Consumo_real_L_ha"] = tomate["Consumo_L_ha"] * tomate["Pasadas_ha"]
tomate["Gasoil_real_L_por_t"] = tomate["Consumo_real_L_ha"] / tomate["Rend_t_ha"]


In [ ]:
#Gasoil por tonelada vs nivel tecnológico

maiz["Consumo_real_L_ha"] = maiz["Consumo_L_ha"] * maiz["Pasadas_ha"]
maiz["Gasoil_real_L_por_t"] = maiz["Consumo_real_L_ha"] / maiz["Rend_t_ha"]

maiz.groupby("tech_level")["Gasoil_real_L_por_t"].mean().plot(
    kind="bar", rot=0
)
plt.ylabel("Gasoil total (L/t)")
plt.title("Maíz – Consumo energético total por nivel tecnológico")
plt.tight_layout()

#plt.savefig("Consumo_energetico_nivel_tecnologico.png", dpi=300, bbox_inches="tight")


plt.show()



In [ ]:
#Productivida operativa

maiz.groupby("tech_level")["Productividad_ha_h"].mean().plot(
    kind="bar", rot=0
)
plt.ylabel("Productividad (ha/h)")
plt.title("Maíz – Productividad operativa")
plt.tight_layout()


plt.show()


In [ ]:
#Uso de insumos
maiz.groupby("tech_level")["Abono_ha"].mean().plot(
    kind="bar", rot=0
)
plt.ylabel("Abono (kg/ha)")
plt.title("Maíz – Uso de abono por nivel tecnológico")
plt.tight_layout()
plt.show()

maiz.groupby("tech_level")["Fitosanitarios_ha"].mean().plot(
    kind="bar", rot=0
)
plt.ylabel("Fitosanitarios (L/ha)")
plt.title("Maíz – Uso de fitosanitarios")
plt.tight_layout()
plt.show()


In [ ]:
#rendimiento por hectárea (solo para maíz)
maiz.groupby("tech_level")["Rend_t_ha"].mean().plot(
    kind="bar", rot=0
)
plt.ylabel("Rendimiento (t/ha)")
plt.title("Maíz – Rendimiento por hectárea")
plt.tight_layout()

plt.savefig("Rendimiento_por_ha.png", dpi=300, bbox_inches="tight")


plt.show()


In [ ]:
df = maiz.dropna(subset=["tech_index", "Gasoil_real_L_por_t"]).copy()

x = df["tech_index"].astype(float).values
y = df["Gasoil_real_L_por_t"].astype(float).values

# Ajuste lineal simple para la recta (solo para visualizar tendencia)
m, b = np.polyfit(x, y, 1)

# Rango para la recta
x_line = np.linspace(x.min(), x.max(), 100)
y_line = m * x_line + b

plt.figure()
plt.scatter(x, y)
plt.plot(x_line, y_line)

plt.title("Maíz – Relación entre IIT y consumo energético específico")
plt.xlabel("Índice de Intensidad Tecnológica (IIT)")
plt.ylabel("Consumo energético específico (L/t)")

plt.tight_layout()

#plt.savefig("Relación_IIT_y_consumo_energético.png", dpi=300, bbox_inches="tight")


plt.show()

In [ ]:
#forest plot. coeficientes del modelo de gasto de gasoil con IC (intervalo de confianza) 95%
#pongo linea vertical en 0, ya que sería que no tiene efecto

params = model.params
conf = model.conf_int()
conf.columns = ["low", "high"]

# Quitar constante para que sea más limpio 
data = pd.concat([params, conf], axis=1)
data.columns = ["coef", "low", "high"]
data = data.drop(index="const", errors="ignore")

# Ordenar por magnitud 
data = data.sort_values("coef", ascending=True)

rename_map = {
    "Corte_tramos": "Corte por tramos",
    "Siembra_variable": "Siembra variable",
    "Abono_variable": "Abono variable"
}
data.index = [rename_map.get(i, i) for i in data.index]

# Plot tipo forest
plt.figure()
y_pos = range(len(data))

plt.hlines(y=y_pos, xmin=data["low"], xmax=data["high"])
plt.plot(data["coef"], y_pos, marker="o", linestyle="None")
plt.axvline(0)

plt.yticks(y_pos, data.index)
plt.xlabel("Coeficiente (efecto marginal) sobre Gasoil_real_L_por_t (L/t)")
plt.title("Maíz – Efecto estimado de cada tecnología (OLS, IC 95%)")

plt.tight_layout()

#plt.savefig("Efecto_tecnología.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
#Comparación Maíz vs Tomate
kpi_compare = pd.DataFrame({
    "Maíz": maiz.groupby("tech_level")["Gasoil_L_por_t"].mean(),
    "Tomate": tomate.groupby("tech_level")["Gasoil_L_por_t"].mean()
})

kpi_compare.plot(kind="bar", rot=0)
plt.ylabel("Gasoil (L/t)")
plt.title("Comparación Maíz vs Tomate – Eficiencia energética")
plt.tight_layout()
plt.show()


In [ ]:
#KPI estrella Tomate
tomate.groupby("tech_level")["Gasoil_real_L_por_t"].mean().plot(
    kind="bar", rot=0
)
plt.ylabel("Gasoil total (L/t)")
plt.title("Tomate – Consumo energético total por nivel tecnológico")
plt.tight_layout()
plt.show()


In [ ]:
#tabla resumen por nivel tecnológico
summary_final = pd.DataFrame({
    # KPI estrella
    "Maíz – Gasoil real (L/t)": maiz.groupby("tech_level")["Gasoil_real_L_por_t"].mean(),
    "Tomate – Gasoil real (L/t)": tomate.groupby("tech_level")["Gasoil_real_L_por_t"].mean(),

    # Productividad
    "Maíz – Productividad (ha/h)": maiz.groupby("tech_level")["Productividad_ha_h"].mean(),
    "Tomate – Productividad (ha/h)": tomate.groupby("tech_level")["Productividad_ha_h"].mean(),

    # Insumos
    "Maíz – Abono (kg/ha)": maiz.groupby("tech_level")["Abono_ha"].mean(),
    "Tomate – Abono (kg/ha)": tomate.groupby("tech_level")["Abono_ha"].mean(),

    "Maíz – Fitosanitarios (L/ha)": maiz.groupby("tech_level")["Fitosanitarios_ha"].mean(),
    "Tomate – Fitosanitarios (L/ha)": tomate.groupby("tech_level")["Fitosanitarios_ha"].mean(),
}).round(2)

summary_final
